# Ray tracing, HMC, and Adam with PyTorch on a Neural Network

General instructions:
1. For neural networks, it's simplest to use the raytracing sampler like you would normally use an optimizer (see example below).  **Please refer to the documentation** for how to set parameters like the likelihood scaling (or target loss delta compared to the best fit result).
2. If you'd like to use exact (instead of approximate) sampling, set the appropriate flag in the training loop call
3. The training loop expects a few extra parameters compared to normal so that it can act like a Markov chain instead of a memory-less optimizer (see example below)

In [1]:
import numpy as np
import scipy as sp
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
%matplotlib inline

## Load the ray tracing / HMC sampler class

In [2]:
#Download raytrace_torch.py locally if it is not already present
#Required if running on Colab
!curl -O https://api.bitbucket.org/2.0/repositories/pbehroozi/ray-tracing-sampler/src/main/raytrace_torch.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 22833  100 22833    0     0  58499      0 --:--:-- --:--:-- --:--:-- 58546


In [3]:
import sys
sys.path.append('..') #if running from repository
from raytrace_torch import Raytracer

## Train and Test Loops

The training loop expects the usual parameters, plus should receive `last_loss` (the last value of the loss function, or 0 if this is the first loop), `optim_type` (one of "Hamil", "Raytracing", or "Adam"), and `likelihood_tolerance` (the change in ln(L) that is tolerable without a Metropolis test; set to 0 for exact sampling, and set to -1 to disable).  The default below is to disable the Metropolis test, to allow you to check different timesteps more easily before deciding on the appropriate loss tolerance (if needed).  As well, the Metropolis test is skipped if the loss drops by `loss_drop_threshold` (default of 0.1 below), so that early progress in the burn-in stage is not subject to a Metropolis test.

In [4]:
import copy
import math

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_loop(dataloader, model, loss_fn, optimizer, last_loss, optim_type, loss_drop_threshold=0.1,
               likelihood_tolerance=-1, save_all_steps=False, save_memory=False):
    models = []
    size = len(dataloader.dataset)
    
    model.eval()
    last_print = 0
    avg_loss = 0

    #save state in case the Metropolis test fails
    if (save_memory):
        torch.save(model.state_dict(), 'model.pth')
        torch.save(optimizer.state_dict(), 'optimizer.pth')
    else:
        model_state_dict = copy.deepcopy(model.state_dict())
        optimizer_state_dict = copy.deepcopy(optimizer.state_dict())

    #save intermediate steps if the user passes a non-bool value to save_all_steps
    #note that save_all_steps will not be a fair MCMC chain, but is instead intended to diagnose
    #mid-epoch issues.
    if (not isinstance(save_all_steps, bool)):
        models.append(nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy())

    #if sampling, let the sampler know to make the first step in the trajectory
    if (optim_type=='Hamil' or optim_type=='Raytracing'):
        scale_likelihood = optimizer.scale_likelihood()
        optimizer.first_step()
        last_luminosity = 0
        if (last_loss > 0):
            last_luminosity = optimizer.ln_luminosity()

    #perform training/sampling for one epoch
    model.train()
    new_loss = 0
    final_loss = 0
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)
        avg_loss += loss.detach().item()*len(X)
        new_loss = loss.detach().item()
        loss.backward()

        #decide if we are in the last batch--if so, set the optimizer to make the final step
        last_batch = 0
        if ((batch+1)*dataloader.batch_size >= size):
            last_batch = 1

        if (batch>0 and last_batch and (optim_type=='Hamil' or optim_type=='Raytracing')):
            optimizer.final_step()

        #perform the sampling/fitting step and reset the gradient
        optimizer.step()
        optimizer.zero_grad()
        
        #save models if requested
        if (not isinstance(save_all_steps, bool)):
            models.append(nn.utils.parameters_to_vector(model.parameters()).detach().cpu().numpy())

        #print out progress
        if ((batch * dataloader.batch_size)>=last_print+2000):
            loss, current = loss.detach().item(), batch * dataloader.batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            last_print = current

    #report average and final (stochastic) losses at end of epoch
    avg_loss /= size
    print(f"Avg train loss: {avg_loss:>8f}")
    print(f"Final train loss: {new_loss:>8f}")
    
    #compute exact loss if required
    if (likelihood_tolerance==0):
        new_loss = 0
        with torch.no_grad():
            for X, y in dataloader:
                pred = model(X)
                new_loss += loss_fn(pred, y).item()
        print(f"Exact train loss: {new_loss:>8f}")
    
    #Calculate Metropolis test probability
    r=torch.rand(1)
    dlnprob = 0
    if ((optim_type == 'Raytracing' or optim_type == 'Hamil') and last_loss>0):
        ln_luminosity = optimizer.ln_luminosity()
        dlnprob = (last_loss-new_loss)*scale_likelihood-(ln_luminosity-last_luminosity)
        #print('Ln Luminosity - Ln Expected Luminosity: %f' % (-dlnprob))
    
    
    #Perform Metropolis test.  This is skipped during the early burn-in stages if the
    # loss drops by at least loss_drop_threshold
    if (last_loss>0 and (loss > last_loss-loss_drop_threshold) and (likelihood_tolerance >= 0) and 
        ((dlnprob < -abs(likelihood_tolerance) and (r[0] > torch.exp(dlnprob))) or (math.isnan(dlnprob)))): 
        if (save_memory):
            model.load_state_dict(torch.load('model.pth'))
            optimizer.load_state_dict(torch.load('optimizer.pth'))
        else:
            model.load_state_dict(model_state_dict)
            optimizer.load_state_dict(optimizer_state_dict)
        optimizer.reverse_momentum()
        new_loss = last_loss
        print("Reset to last step (with momentum reversal) after Metropolis test")
    else:
        if (not isinstance(save_all_steps, bool)):
            save_all_steps.extend(models)
    return new_loss


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()

    test_loss /= num_batches
    print(f"Avg test loss: {test_loss:>8f} \n")
    return test_loss

## Example application for training a MLP to emulate a slower code (i.e., predicting stellar masses for halos in the UniverseMachine)

### Load / download training data

In [1]:
#Load and process halo data if it is not already present in um.npz
import subprocess

def process_data():
    print("Downloading and extracting halo data (359M)...")
    try:
        subprocess.run(["wget", "https://halos.as.arizona.edu/behroozi/Raytracing/raytracing_data.tar.gz"], capture_output=True)
    except:
        subprocess.run(["curl", "-O", "https://halos.as.arizona.edu/behroozi/Raytracing/raytracing_data.tar.gz"], capture_output=True)        
    subprocess.run(["tar", "xzvf", "raytracing_data.tar.gz"])
    print("Loading halo data...")
    data = np.genfromtxt("joined.txt")
    halo_props = np.genfromtxt("joined_halo_info.txt")
    print("Processing halo data (can take a minute or two)...")
    halo_props_dict = {}
    for x in range(len(halo_props)):
        halo_props_dict[str(int(halo_props[x][1]))] = x
    
    joined_props = []
    for x in data:
        id = halo_props_dict.get(str(int(x[1])),-1)
        if (id>-1):
            joined_props.append(list(x)+list(halo_props[id]))
    processed_data = np.array(joined_props)
    hlist_props = np.array([10,11,12,13,16,26,35,37,38,39,40,41,42,43,44,45,46,47,51,52,56,57,58])+29
    
    for x in [12,13,14,15,16,21,22,23,24,25,26,27]:
        tmp = processed_data[:,x]
        tmp[tmp<=0] = 1e-5
        processed_data[:,x] = np.log10(tmp)
    
    for x in hlist_props:
        tmp = processed_data[:,x]
        tmp[tmp<=0] = 1e-5
        processed_data[:,x] = np.log10(tmp)
    
    input_cols = [0,12,13,19,20]
    
    input = processed_data[:,input_cols]
    output = processed_data[:,[21]]
    print("Saving halo data...")
    np.savez("um.npz", allow_pickle=False, input=input, output=output)
    print("The processed data is cached in um.npz so sucessive loads will be faster.")


In [7]:
#Try loading the preprocessed data; otherwise download
try:
    with open("um.npz", "r") as file:
        file.close()
except FileNotFoundError:
    process_data()
    
data = np.load("um.npz")
input = data['input']
output = data['output']
if (len(input) != 987214 and len(output) != 987214):
    raise RuntimeError("Corrupted um.npz.  Delete file and rerun process_data() to get a fresh one.")
print("Processed data.  First entries in input and output arrays:")
print(input[0])
print(output[0])

joined_halo_info.txt
joined.txt
Loading halo data...
Processing halo data (can take a minute or two)...
Saving halo data...
The processed data is cached in um.npz so sucessive loads will be faster.
Processed data.  First entries in input and output arrays:
[ 0.323935   11.25382244  2.09173727 -0.796671   -0.464268  ]
[9.77974075]


### Data preprocessing

In [8]:
#split into training and test sets
output_train, output_test, input_train, input_test = \
    train_test_split(output, input, test_size=0.2, random_state=42, shuffle=False)

print(output_train[0])
print(output_test[0])

print("Training set length: ", len(output_train))
print("Test set length: ", len(output_test))

[9.77974075]
[10.20493352]
Training set length:  789771
Test set length:  197443


In [9]:
#rescale input and output sets to zero mean and unit variance
def rescale(train, test):
    shape = np.shape(train)
    new_train = np.copy(train)
    new_test = np.copy(test)
    if (len(shape)==1):
        avg = np.mean(train)
        std = np.std(train)
        new_train = (new_train-avg)/std
        new_test = (new_test-avg)/std
        return (new_train,new_test,avg, std)
    else:
        avgs = []
        stds = []
        for i in range(0,shape[1]):
            avg = np.mean(train[:,i])
            std = np.std(train[:,i])
            new_train[:,i] = (new_train[:,i]-avg)/std
            new_test[:,i] = (new_test[:,i]-avg)/std
            avgs.append(avg)
            stds.append(std)
        return (new_train,new_test,avgs, stds)

#scale training and test sets
output_train_scaled,output_test_scaled,output_mean, output_std = rescale(output_train, output_test)
print("Stellar Mass Mean / Std: ", output_mean, output_std)

input_train_scaled,input_test_scaled,input_means, input_stds = rescale(input_train, input_test)
print("Input Mean / Std: ", input_means, input_stds)

Stellar Mass Mean / Std:  [np.float64(9.101696066823852)] [np.float64(1.779604913439086)]
Input Mean / Std:  [np.float64(0.6111613207955218), np.float64(11.524401499646304), np.float64(2.1121893293814002), np.float64(0.0037095644458963414), np.float64(-0.0010509357864494895)] [np.float64(0.22150280660027322), np.float64(1.0716531998831145), np.float64(0.3244750126159838), np.float64(1.0001073555839222), np.float64(1.001382648623918)]


In [10]:
print("First input training vector (unscaled): ", input_train[0])
print("First input training vector (scaled): ", input_train_scaled[0])
print("First training output (unscaled): ", output_train[0])
print("First training output (scaled): ",output_train_scaled[0])

First input training vector (unscaled):  [ 0.323935   11.25382244  2.09173727 -0.796671   -0.464268  ]
First input training vector (scaled):  [-1.29671639 -0.25248752 -0.06303122 -0.80029465 -0.46257748]
First training output (unscaled):  [9.77974075]
First training output (scaled):  [0.38100855]


### Convert data to be useful for PyTorch 

In [36]:
# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

#If on Colab, I recommend using the CPUs or GPUs
#For some reason (possibly compiling/lazy execution), attempting to use TPUs
#takes a very long time with the below code
#Happy to work with any Googlers to fix this
#import torch_xla
#import torch_xla.core.xla_model as xm
#device = xm.xla_device()

print(f"Using {device} device")

Using cuda device


In [13]:
#Convert data to dataloader
def torch_tensor(x):
    return torch.from_numpy(x).to(torch.float32).to(device)

torch.manual_seed(6822790577)
torch_train_input = torch_tensor(input_train_scaled)
torch_train_output = torch_tensor(output_train_scaled)
torch_test_input = torch_tensor(input_test_scaled)
torch_test_output = torch_tensor(output_test_scaled)
print("Torch data on device:", torch_train_input.device)

training_data = torch.utils.data.TensorDataset(torch_train_input, torch_train_output)
test_data = torch.utils.data.TensorDataset(torch_test_input, torch_test_output)

train_dataloader = torch.utils.data.DataLoader(training_data, batch_size=20000, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=20000, shuffle=False)

train_dataloader_1000 = torch.utils.data.DataLoader(training_data, batch_size=1000, shuffle=True)
train_dataloader_5000 = torch.utils.data.DataLoader(training_data, batch_size=5000, shuffle=True)
train_dataloader_full = torch.utils.data.DataLoader(training_data, batch_size=len(training_data), shuffle=True)
(test_subsample, rest) = torch.utils.data.random_split(test_data, [0.05, 0.95])
test_dataloader_subsampled = torch.utils.data.DataLoader(test_subsample, batch_size=2000, shuffle=False)

Torch data on device: cuda:0


### Build a simple fully-connected (MLP) network with SELU activations:

In [16]:
input_nodes = len(input_train[0])
output_nodes = len(output_train[0])
loss_fn = nn.MSELoss()

In [17]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [18]:
from collections import OrderedDict

def gen_SELU(arch):
    layers = OrderedDict()
    for x in range(len(arch)-2):
        layers['layer'+str(x)] = nn.Linear(arch[x], arch[x+1])
        layers['rl'+str(x)] = nn.SELU()
    layers['output'] = nn.Linear(arch[-2], arch[-1])
    return nn.Sequential(layers)

In [19]:
model = gen_SELU([input_nodes,8,16,24,32,output_nodes]).to(device)
print(count_parameters(model))

1433


In [20]:
#Helper code for copying / adding / measuring models in weight space

def parameters_norm(m):
    norm2 = 0
    for param in m.parameters():
        norm2 += torch.sum(torch.square(param))
    return torch.sqrt(norm2.cpu())
    
def parameters_add(m1,m2,alpha):
    with torch.no_grad():
        for param1,param2 in zip(m1.parameters(), m2.parameters()):
            param1.add_(param2, alpha=alpha)

def parameters_mul(m,alpha):
    parameters_add(m, m, alpha-1)

def parameters_copy(m1, m2):
    parameters_add(m1, m1, -1)
    parameters_add(m1, m2, 1)

### Code that helps runs many MCMC chains / optimizers and saves results

`batch_models()` is the main workhorse here, and requires at least three parameters: 1) `optim_type`, set as for the training loop above; 2) `mul`, which determines the scaling of the initial network weights; and 3) `models`, which determines how many MCMC chains to run or independent models to fit.  The code also has parameters to set the threshold log-likelihood for Metropolis tests (the same as the training loop above), to set training / sampling parameters like the timestep (via `dt`, which is also used as the learning rate for Adam), the momentum refresh rate (via `refresh_momentum`), and the likelihood scaling (via `scale_likelihood`).

In [22]:
#Helper code for running many MCMC chains / optimizers and saving results
def save_test_results(model, dataloader, filename):
     with open(filename, "w") as f:
        predicted = []
        with torch.no_grad():
            for inputs, labels in dataloader:
                outputs = model(inputs)-labels
                predicted += outputs.squeeze().cpu()
        predicted = np.array(predicted)
        f.write(" ".join(map(str, predicted)) + "\n")
        f.write('#test_loss: '+str((predicted**2).mean())+"\n")

def init_optimizer(parameters, dt, scale_likelihood, optim_type, refresh_rate):
    if (optim_type=='Hamil'):
        optimizer = Raytracer(parameters, dt=dt, scale_likelihood=scale_likelihood, stochastic_hmc=1, refresh_rate=refresh_rate)
    elif (optim_type=='Raytracing'):
        optimizer = Raytracer(parameters, dt=dt, scale_likelihood=scale_likelihood, refresh_rate=refresh_rate)
    else:
        optimizer = torch.optim.Adam(parameters, lr=dt)
    return optimizer


#This is the code that generates many fresh models and then runs fitting / sampling as requested by the user
def batch_models(optim_type, mul, num_models, dt=0.001, threshold_dlnl=80, epochs=15, starting_models=None, 
                 refresh_rate=1, train_dl=train_dataloader, test_dl=test_dataloader, save_all_steps=False, 
                 save_tests=False, save_tests_dl=False, refresh_momentum=0, scale_likelihood=5000,
                 save_memory=False, skip=0):
    models = []
    norms = []
    starting_model = gen_SELU([input_nodes,8,16,24,32,output_nodes]).to(device)
    distances = []
    model_test_loss = []
    network_output_list = []
    pred_halo_1 = []
    pred_halo_2 = []
    starting_norms = []
    if (save_tests_dl==False):
        save_tests_dl=test_dl
    for i in range(num_models):
        models.append(gen_SELU([input_nodes,8,16,24,32,output_nodes]).to(device))
        if (starting_models != None):
            parameters_copy(models[i],starting_models['models'][i])
        parameters_mul(models[i], mul)
        parameters_copy(starting_model, models[i])
        starting_norms.append(parameters_norm(starting_model))
        optimizer = init_optimizer(models[i].parameters(), dt, scale_likelihood, optim_type, refresh_rate)
        last_loss = 0
        test_loss = 0
        saved_steps = False
        for x in range(epochs):
            if (save_all_steps):
                saved_steps=[]
            last_loss = train_loop(train_dl,models[i], loss_fn, optimizer, last_loss, optim_type, 
                                   threshold_dlnl, save_all_steps=saved_steps, save_memory=save_memory)
            if (save_all_steps):
                with open(save_all_steps+'_m'+str(i+skip)+'_e'+str(x)+'.params', "w") as f:
                    for y in saved_steps:
                        f.write(" ".join(map(str, y)) + "\n")
                    f.write('#train_loss: '+str(last_loss)+"\n")                        
            if (save_tests):
                save_test_results(models[i], save_tests_dl, save_tests+'_m'+str(i+skip)+'_e'+str(x)+'.params')
            if (torch.rand(1)<refresh_momentum):
                optimizer = init_optimizer(models[i].parameters(), dt, scale_likelihood, optim_type, refresh_rate)
            print("Param norm: ",parameters_norm(models[i]))
        test_loss = test_loop(test_dl, models[i], loss_fn)
        norms.append(parameters_norm(models[i]))
        parameters_add(starting_model, models[i], -1)
        distances.append(parameters_norm(starting_model))
        model_test_loss.append(test_loss)        
        network_output_scaled = models[i](torch.from_numpy(input_test_scaled).to(torch.float32).to(device)).numpy(force=True)
        network_outputs = (network_output_scaled*output_std)+output_mean
        network_output_list.append(network_outputs)
        pred_halo_1.append(network_outputs[100000,0])
        pred_halo_2.append(network_outputs[100001,0])
        print("Mean abs. SM error: ", np.mean(np.abs((output_test[:,0]-network_outputs[:,0]))))
        print("Std. Dev. SM: ", np.sqrt(np.mean(np.abs((output_test[:,0]-network_outputs[:,0])**2))))
        print("Finished model ",i)
    return ({'models':models, 'norms':norms, 'distances':distances, 'test_loss':model_test_loss, 'network_outputs':network_output_list,
            'pred_halo_1':pred_halo_1, 'pred_halo_2':pred_halo_2, 'optim_type':optim_type, 'mul':mul, 'num_models':num_models,
            'starting_norms':starting_norms})


## Run ray tracing, Adam, and HMC

The following code implements the example in Section 3.2 of the ray tracing paper, with parameters described in Appendix E.1.  It can take several days to run all of these sequentially, so you may consider reducing the number of epochs and model counts to complete a simpler test run faster.

In [35]:
sampling_epochs = 1000
sampling_models = 10

fitting_epochs = 50
fitting_models = 1000

sample_trajectory_epochs = 50000

In [28]:
#Run ray tracing sampling on 10 independent starting positions for 1000 epochs each
for dt in (['2e-4']):
    torch.manual_seed(6822790577)
    ray_refresh_burn_12 = batch_models('Raytracing',3,sampling_models,refresh_rate=5, dt=float(dt), 
                                       epochs=sampling_epochs, 
                                       save_all_steps='fan_selu_ray_r5_'+str(dt), 
                                       save_tests='fan_selu_ray_test_r5_'+str(dt), 
                                       save_tests_dl=test_dataloader_subsampled, threshold_dlnl=50)

loss: 20.113274  [40000/789771]
loss: 17.572428  [80000/789771]
loss: 15.596098  [120000/789771]
loss: 13.602337  [160000/789771]
loss: 12.492870  [200000/789771]
loss: 11.141303  [240000/789771]
loss: 10.163843  [280000/789771]
loss: 9.418479  [320000/789771]
loss: 8.795369  [360000/789771]
loss: 8.219386  [400000/789771]
loss: 7.757770  [440000/789771]
loss: 7.091486  [480000/789771]
loss: 6.638227  [520000/789771]
loss: 6.109479  [560000/789771]
loss: 5.776255  [600000/789771]
loss: 5.329744  [640000/789771]
loss: 5.012988  [680000/789771]
loss: 4.783276  [720000/789771]
loss: 4.469096  [760000/789771]
loss: 4.143066  [789771/789771]
Avg train loss: 9.534130
Final train loss: 4.143066
Param norm:  tensor(15.9076, grad_fn=<SqrtBackward0>)
loss: 4.012884  [40000/789771]
loss: 3.719648  [80000/789771]
loss: 3.576946  [120000/789771]
loss: 3.370118  [160000/789771]
loss: 3.208508  [200000/789771]
loss: 3.065241  [240000/789771]
loss: 2.774362  [280000/789771]
loss: 2.626216  [320000/789

In [34]:
#Generate a sample trajectory through the loss surface with ray tracing and no refresh
try:
     for dt in (['2e-5']):
        torch.manual_seed(6822790577)
        ray_norefresh = batch_models('Raytracing',1,1,refresh_rate=0, starting_models=ray_refresh_burn_12,
                                     dt=float(dt), epochs=sample_trajectory_epochs, 
                                     save_all_steps='fan_selu_ray_r0_'+str(dt), 
                                     save_tests='fan_selu_ray_test_r0_'+str(dt), 
                                     save_tests_dl=test_dataloader_subsampled, threshold_dlnl=-1)
except NameError:
    print("The preceding cell needs to run to completion first.")

loss: 0.072814  [40000/789771]
loss: 0.074975  [80000/789771]
loss: 0.074402  [120000/789771]
loss: 0.075050  [160000/789771]
loss: 0.074342  [200000/789771]
loss: 0.075979  [240000/789771]
loss: 0.075333  [280000/789771]
loss: 0.073339  [320000/789771]
loss: 0.075451  [360000/789771]
loss: 0.076196  [400000/789771]
loss: 0.076103  [440000/789771]
loss: 0.076443  [480000/789771]
loss: 0.072412  [520000/789771]
loss: 0.072721  [560000/789771]
loss: 0.072716  [600000/789771]
loss: 0.073814  [640000/789771]
loss: 0.074906  [680000/789771]
loss: 0.074757  [720000/789771]
loss: 0.074074  [760000/789771]
loss: 0.071862  [789771/789771]
Avg train loss: 0.074497
Final train loss: 0.071862
Param norm:  tensor(15.6631, grad_fn=<SqrtBackward0>)
Avg test loss: 0.075410 

Mean abs. SM error:  0.3595413079495489
Std. Dev. SM:  0.48859205995509475
Finished model  0


In [ ]:
#Run stochastic HMC sampling on 10 independent starting positions for 1000 epochs each
#Burn in will run for fitting_epochs
for dt in (['2e-4']):
    torch.manual_seed(6822790577)
    hmc_refresh_burn_12 = batch_models('Hamil',3,sampling_models,refresh_rate=50, dt=float(dt), 
                                       epochs=fitting_epochs, 
                                       save_all_steps='hmc_burnin_'+dt, save_tests='hmc_burnin_test_'+dt,  
                                       threshold_dlnl=-1, save_tests_dl=test_dataloader_subsampled)
    hmc_refresh_12 = batch_models('Hamil',1,sampling_models,starting_models=hmc_refresh_burn_12, refresh_rate=5, 
                                  dt=float(dt), epochs=sampling_epochs, save_all_steps='hmc_'+dt, 
                                  save_tests='hmc_test_'+dt, threshold_dlnl=50,
                                  save_tests_dl=test_dataloader_subsampled)

In [ ]:
#Run Adam fitting on 1000 independent starting positions for 50 epochs each
torch.manual_seed(6822790578)
adam_refresh_12 = batch_models('Adam',3,fitting_models, epochs = fitting_epochs, dt=0.02,  
                                   save_tests='adam_test', save_tests_dl=test_dataloader_subsampled)

loss: 1.385404  [40000/789771]
loss: 1.151418  [80000/789771]
loss: 0.998325  [120000/789771]
loss: 0.842532  [160000/789771]
loss: 0.683289  [200000/789771]
loss: 0.608591  [240000/789771]
loss: 0.539565  [280000/789771]
loss: 0.468483  [320000/789771]
loss: 0.410951  [360000/789771]
loss: 0.375906  [400000/789771]
loss: 0.352589  [440000/789771]
loss: 0.330802  [480000/789771]
loss: 0.305915  [520000/789771]
loss: 0.294340  [560000/789771]
loss: 0.283066  [600000/789771]
loss: 0.279025  [640000/789771]
loss: 0.258440  [680000/789771]
loss: 0.254711  [720000/789771]
loss: 0.245771  [760000/789771]
loss: 0.231285  [789771/789771]
Avg train loss: 0.540436
Param norm:  tensor(15.6355, grad_fn=<SqrtBackward0>)
loss: 0.229315  [40000/789771]
loss: 0.217695  [80000/789771]
loss: 0.209278  [120000/789771]
loss: 0.202470  [160000/789771]
loss: 0.191294  [200000/789771]
loss: 0.188030  [240000/789771]
loss: 0.186707  [280000/789771]
loss: 0.179640  [320000/789771]
loss: 0.174837  [360000/78977

loss: 0.049727  [440000/789771]
loss: 0.051475  [480000/789771]
loss: 0.050986  [520000/789771]
loss: 0.051059  [560000/789771]
loss: 0.050220  [600000/789771]
loss: 0.049817  [640000/789771]
loss: 0.049285  [680000/789771]
loss: 0.049844  [720000/789771]
loss: 0.049732  [760000/789771]
loss: 0.051152  [789771/789771]
Avg train loss: 0.051038
Param norm:  tensor(14.9089, grad_fn=<SqrtBackward0>)
loss: 0.049536  [40000/789771]
loss: 0.048839  [80000/789771]
loss: 0.050465  [120000/789771]
loss: 0.050214  [160000/789771]
loss: 0.050341  [200000/789771]
loss: 0.048549  [240000/789771]
loss: 0.050516  [280000/789771]
loss: 0.050499  [320000/789771]
loss: 0.049022  [360000/789771]
loss: 0.048822  [400000/789771]
loss: 0.048878  [440000/789771]
loss: 0.049805  [480000/789771]
loss: 0.050077  [520000/789771]
loss: 0.048663  [560000/789771]
loss: 0.049151  [600000/789771]
loss: 0.048862  [640000/789771]
loss: 0.048149  [680000/789771]
loss: 0.048158  [720000/789771]
loss: 0.047989  [760000/789

Param norm:  tensor(14.6141, grad_fn=<SqrtBackward0>)
loss: 0.042690  [40000/789771]
loss: 0.043434  [80000/789771]
loss: 0.041577  [120000/789771]
loss: 0.042175  [160000/789771]
loss: 0.042815  [200000/789771]
loss: 0.042353  [240000/789771]
loss: 0.042655  [280000/789771]
loss: 0.043358  [320000/789771]
loss: 0.043170  [360000/789771]
loss: 0.043128  [400000/789771]
loss: 0.043550  [440000/789771]
loss: 0.042499  [480000/789771]
loss: 0.041584  [520000/789771]
loss: 0.042848  [560000/789771]
loss: 0.041270  [600000/789771]
loss: 0.043031  [640000/789771]
loss: 0.042168  [680000/789771]
loss: 0.042587  [720000/789771]
loss: 0.042004  [760000/789771]
loss: 0.042952  [789771/789771]
Avg train loss: 0.042693
Param norm:  tensor(14.5906, grad_fn=<SqrtBackward0>)
loss: 0.041547  [40000/789771]
loss: 0.042220  [80000/789771]
loss: 0.041792  [120000/789771]
loss: 0.042070  [160000/789771]
loss: 0.043053  [200000/789771]
loss: 0.041064  [240000/789771]
loss: 0.041883  [280000/789771]
loss: 0

loss: 0.058809  [280000/789771]
loss: 0.060748  [320000/789771]
loss: 0.061841  [360000/789771]
loss: 0.059690  [400000/789771]
loss: 0.060512  [440000/789771]
loss: 0.060306  [480000/789771]
loss: 0.060218  [520000/789771]
loss: 0.058927  [560000/789771]
loss: 0.060331  [600000/789771]
loss: 0.058862  [640000/789771]
loss: 0.058936  [680000/789771]
loss: 0.059390  [720000/789771]
loss: 0.057133  [760000/789771]
loss: 0.059987  [789771/789771]
Avg train loss: 0.059987
Param norm:  tensor(15.4190, grad_fn=<SqrtBackward0>)
loss: 0.060843  [40000/789771]
loss: 0.057695  [80000/789771]
loss: 0.058653  [120000/789771]
loss: 0.059086  [160000/789771]
loss: 0.056722  [200000/789771]
loss: 0.056313  [240000/789771]
loss: 0.057836  [280000/789771]
loss: 0.057714  [320000/789771]
loss: 0.058856  [360000/789771]
loss: 0.058256  [400000/789771]
loss: 0.059559  [440000/789771]
loss: 0.057765  [480000/789771]
loss: 0.057753  [520000/789771]
loss: 0.057149  [560000/789771]
loss: 0.058548  [600000/789

loss: 0.047214  [720000/789771]
loss: 0.046256  [760000/789771]
loss: 0.044838  [789771/789771]
Avg train loss: 0.045974
Param norm:  tensor(15.1331, grad_fn=<SqrtBackward0>)
loss: 0.044286  [40000/789771]
loss: 0.044447  [80000/789771]
loss: 0.046164  [120000/789771]
loss: 0.045459  [160000/789771]
loss: 0.045178  [200000/789771]
loss: 0.045961  [240000/789771]
loss: 0.044792  [280000/789771]
loss: 0.045846  [320000/789771]
loss: 0.045277  [360000/789771]
loss: 0.045935  [400000/789771]
loss: 0.046281  [440000/789771]
loss: 0.044425  [480000/789771]
loss: 0.044948  [520000/789771]
loss: 0.045280  [560000/789771]
loss: 0.044467  [600000/789771]
loss: 0.045869  [640000/789771]
loss: 0.044141  [680000/789771]
loss: 0.045713  [720000/789771]
loss: 0.043906  [760000/789771]
loss: 0.044638  [789771/789771]
Avg train loss: 0.045359
Param norm:  tensor(15.1111, grad_fn=<SqrtBackward0>)
loss: 0.045530  [40000/789771]
loss: 0.045670  [80000/789771]
loss: 0.044922  [120000/789771]
loss: 0.045286

...

loss: 0.067212  [720000/789771]
loss: 0.066625  [760000/789771]
loss: 0.068183  [789771/789771]
Avg train loss: 0.067274
Param norm:  tensor(15.6812, grad_fn=<SqrtBackward0>)
loss: 0.066627  [40000/789771]
loss: 0.068401  [80000/789771]
loss: 0.065915  [120000/789771]
loss: 0.067624  [160000/789771]
loss: 0.066486  [200000/789771]
loss: 0.065122  [240000/789771]
loss: 0.067633  [280000/789771]
loss: 0.065924  [320000/789771]
loss: 0.066545  [360000/789771]
loss: 0.066444  [400000/789771]
loss: 0.064271  [440000/789771]
loss: 0.066597  [480000/789771]
loss: 0.067094  [520000/789771]
loss: 0.063689  [560000/789771]
loss: 0.063960  [600000/789771]
loss: 0.065565  [640000/789771]
loss: 0.065973  [680000/789771]
loss: 0.064715  [720000/789771]
loss: 0.063818  [760000/789771]
loss: 0.065351  [789771/789771]
Avg train loss: 0.065898
Param norm:  tensor(15.6726, grad_fn=<SqrtBackward0>)
loss: 0.064437  [40000/789771]
loss: 0.065128  [80000/789771]
loss: 0.065676  [120000/789771]
loss: 0.063165

loss: 0.064652  [120000/789771]
loss: 0.064382  [160000/789771]
loss: 0.064264  [200000/789771]
loss: 0.065387  [240000/789771]
loss: 0.064001  [280000/789771]
loss: 0.063334  [320000/789771]
loss: 0.062802  [360000/789771]
loss: 0.064972  [400000/789771]
loss: 0.062735  [440000/789771]
loss: 0.064321  [480000/789771]
loss: 0.062653  [520000/789771]
loss: 0.064546  [560000/789771]
loss: 0.063015  [600000/789771]
loss: 0.062842  [640000/789771]
loss: 0.062263  [680000/789771]
loss: 0.063210  [720000/789771]
loss: 0.063800  [760000/789771]
loss: 0.063796  [789771/789771]
Avg train loss: 0.063879
Param norm:  tensor(15.4861, grad_fn=<SqrtBackward0>)
loss: 0.063477  [40000/789771]
loss: 0.062428  [80000/789771]
loss: 0.060711  [120000/789771]
loss: 0.063442  [160000/789771]
loss: 0.062027  [200000/789771]
loss: 0.060985  [240000/789771]
loss: 0.061812  [280000/789771]
loss: 0.063077  [320000/789771]
loss: 0.062005  [360000/789771]
loss: 0.061432  [400000/789771]
loss: 0.061789  [440000/789

loss: 0.049623  [520000/789771]
loss: 0.048814  [560000/789771]
loss: 0.049655  [600000/789771]
loss: 0.049112  [640000/789771]
loss: 0.048220  [680000/789771]
loss: 0.048428  [720000/789771]
loss: 0.049093  [760000/789771]
loss: 0.048101  [789771/789771]
Avg train loss: 0.049121
Param norm:  tensor(15.0740, grad_fn=<SqrtBackward0>)
loss: 0.048497  [40000/789771]
loss: 0.049708  [80000/789771]
loss: 0.047547  [120000/789771]
loss: 0.048132  [160000/789771]
loss: 0.050344  [200000/789771]
loss: 0.048691  [240000/789771]
loss: 0.050264  [280000/789771]
loss: 0.049291  [320000/789771]
loss: 0.047663  [360000/789771]
loss: 0.047118  [400000/789771]
loss: 0.048787  [440000/789771]
loss: 0.046401  [480000/789771]
loss: 0.049709  [520000/789771]
loss: 0.048618  [560000/789771]
loss: 0.047559  [600000/789771]
loss: 0.048392  [640000/789771]
loss: 0.048167  [680000/789771]
loss: 0.047537  [720000/789771]
loss: 0.048965  [760000/789771]
loss: 0.048230  [789771/789771]
Avg train loss: 0.048421
Pa

loss: 0.187786  [760000/789771]
loss: 0.185723  [789771/789771]
Avg train loss: 0.203984
Param norm:  tensor(16.0182, grad_fn=<SqrtBackward0>)
loss: 0.185545  [40000/789771]
loss: 0.184309  [80000/789771]
loss: 0.183406  [120000/789771]
loss: 0.181906  [160000/789771]
loss: 0.180395  [200000/789771]
loss: 0.180206  [240000/789771]
loss: 0.176597  [280000/789771]
loss: 0.179106  [320000/789771]
loss: 0.180173  [360000/789771]
loss: 0.173069  [400000/789771]
loss: 0.177809  [440000/789771]
loss: 0.171022  [480000/789771]
loss: 0.174987  [520000/789771]
loss: 0.168974  [560000/789771]
loss: 0.170537  [600000/789771]
loss: 0.168101  [640000/789771]
loss: 0.167003  [680000/789771]
loss: 0.166516  [720000/789771]
loss: 0.161772  [760000/789771]
loss: 0.163771  [789771/789771]
Avg train loss: 0.175374
Param norm:  tensor(15.9922, grad_fn=<SqrtBackward0>)
loss: 0.164472  [40000/789771]
loss: 0.161911  [80000/789771]
loss: 0.163339  [120000/789771]
loss: 0.161668  [160000/789771]
loss: 0.162363

loss: 0.081892  [280000/789771]
loss: 0.081582  [320000/789771]
loss: 0.081323  [360000/789771]
loss: 0.081119  [400000/789771]
loss: 0.080638  [440000/789771]
loss: 0.080014  [480000/789771]
loss: 0.081843  [520000/789771]
loss: 0.082738  [560000/789771]
loss: 0.080442  [600000/789771]
loss: 0.079658  [640000/789771]
loss: 0.080117  [680000/789771]
loss: 0.079759  [720000/789771]
loss: 0.079929  [760000/789771]
loss: 0.078868  [789771/789771]
Avg train loss: 0.081047
Param norm:  tensor(15.8253, grad_fn=<SqrtBackward0>)
loss: 0.078793  [40000/789771]
loss: 0.078137  [80000/789771]
loss: 0.078810  [120000/789771]
loss: 0.079246  [160000/789771]
loss: 0.079055  [200000/789771]
loss: 0.078758  [240000/789771]
loss: 0.078990  [280000/789771]
loss: 0.077988  [320000/789771]
loss: 0.080030  [360000/789771]
loss: 0.077762  [400000/789771]
loss: 0.079044  [440000/789771]
loss: 0.076876  [480000/789771]
loss: 0.077364  [520000/789771]
loss: 0.076593  [560000/789771]
loss: 0.076869  [600000/789

In [ ]:
#Compute average distance traveled compared to the starting norm with Adam
try:
    m = 0
    c = 0
    for i in range(len(adam_refresh_12['distances'])):
        d = adam_refresh_12['distances'][i]
        n = adam_refresh_12['starting_norms'][i]
        m += d/n
        c  += 1.0
    print(m/c)

except NameError:
    print("The preceding cell needs to run to completion first.") 